In [1]:
import pandas as pd

complaints = pd.read_csv('student_loan_complaints.csv')
edu = pd.read_csv('edu_clean.csv')

edu['zipcode'] = edu['Geographic Area Name'].str.extract(r'(\d{5})')
edu = edu.rename(columns={
    'Percent!!Estimate!!Population 25 years and over!!Bachelor\'s degree': 'bachelors_pct'
})
edu['bachelors_pct'] = pd.to_numeric(edu['bachelors_pct'], errors='coerce')

complaints['zipcode'] = complaints['zipcode'].astype(str).str.zfill(5).str[:5]

merged = complaints.merge(edu[['zipcode', 'bachelors_pct']], on='zipcode', how='left')

analysis_df = merged[merged['company_response_to_consumer'].isin([
    'Closed with monetary relief', 
    'Closed with explanation'
])].copy()

analysis_df['response_type'] = analysis_df['company_response_to_consumer'].apply(
    lambda x: 'Monetary Relief' if x == 'Closed with monetary relief' else 'Explanation'
)

analysis_df = analysis_df.dropna(subset=['bachelors_pct'])

analysis_df['edu_quartile'] = pd.qcut(
    analysis_df['bachelors_pct'], 
    q=4, 
    labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)']
)

grouped = analysis_df.groupby(['edu_quartile', 'response_type']).size().unstack(fill_value=0)
grouped['Total'] = grouped.sum(axis=1)
grouped['Monetary Relief Rate (%)'] = (grouped['Monetary Relief'] / grouped['Total'] * 100).round(2)

print(grouped)

response_type  Explanation  Monetary Relief  Total  Monetary Relief Rate (%)
edu_quartile                                                                
Q1 (Lowest)           2535              191   2726                      7.01
Q2                    2499              202   2701                      7.48
Q3                    2496              215   2711                      7.93
Q4 (Highest)          2449              248   2697                      9.20
